## Train a linear probe on Raven unit embeddings
---
Embed exported Raven units with Perch2, optionally cluster background clips, and train a linear classifier using folder names as class labels.

Author: Danelle Cline dcline@mbari.org

### Set paths
Choose the config YAML, dataset directory, and output directory that contains (or will contain) exported Raven unit clips.

In [ ]:
from pathlib import Path
config_yaml_path =  "../config.yaml"
output_path = Path("output_hb")
dataset_path = Path("dataset_hb")
output_path.mkdir(parents=True, exist_ok=True)

### Load configuration
Load and verify the config. Later cells use it for Perch2 window settings and model output paths.

In [ ]:
from stm.config import Config
try:
    config = Config(config_yaml_path, output_path)
    config.verify()
except Exception as e:
    print(e)

### Embed Raven units
Run the Perch2 ONNX model over cached unit clips and collect embeddings for classification.

In [ ]:
# Compute embeddings from the Perch2 ONNX model
from pathlib import Path
from stm.cache import UnitCacheKey
from stm.config import Config
from stm.embed import Embedding
from stm.classify import build_model

config = Config(dataset_path, output_path)
emb = Embedding(UnitCacheKey.from_config(config), Path("perch_v2.onnx"))
embeddings = emb.run()

### Cluster background clips
Density-cluster background unit embeddings so similar noise/background sounds share a cluster label. `assign_noise=True` assigns every file a cluster number.

In [ ]:
from stm.cluster.density import cluster_directory

c = cluster_directory(
    output_path / "units" / "MARS_20161221_000046_SongSession_32kHz_HPF5Hz"/ "background",
    # output_path / "units" / "sequence"/ "background",
    model_path="perch_v2.onnx",
    config=config,          # optional; uses perch window settings
    assign_noise=True,      # default: every file gets a cluster number
)
print(f"Found {c.n_clusters} background cluster(s)")

### Train and sanity-check the linear probe
Build a linear model from unique unit-folder labels, train on the embeddings, save the checkpoint, and print training accuracy. This in-sample check is a sanity test, not a held-out evaluation.

In [ ]:
import torch
from stm.train_utils import labels_to_one_hot

clips = sorted((config.output_path / "units").rglob("*.wav"))
labels = [clip.parent.name for clip in clips]
unique_labels = list(set(labels))
one_hot_labels, class_to_index = labels_to_one_hot(labels)

model = build_model(len(unique_labels))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X = torch.from_numpy(embeddings).to(device)
y = one_hot_labels.to(device)
n = len(X)

epochs = 1000
batch_size = 64

for epoch in range(epochs):
    perm = torch.randperm(n)
    epoch_loss = 0.0
    for lo in range(0, n, batch_size):
        idx = perm[lo : lo + batch_size]
        epoch_loss += model.train_step(X[idx], y[idx]) * len(idx)
    epoch_loss /= n
    if epoch % max(1, epochs // 10) == 0 or epoch == epochs - 1:
        print(f"epoch {epoch:4d}  loss {epoch_loss:.5f}")

model.save(config.model_path / "linear_model.pt", classes=sorted(set(labels)))

# Test the model against its own data; this is a contrived example for sanity check
# Best practice is to have a separate test set. This should return a 100.0% training accuracy
idx_to_class = {i: name for name, i in class_to_index.items()}
pred_idx = model.predict_proba(X).argmax(dim=1).cpu().numpy()
pred_labels = [idx_to_class[int(i)] for i in pred_idx]
n_correct = sum(p == t for p, t in zip(pred_labels, labels))
print(f"train accuracy: {n_correct}/{len(labels)} ({n_correct / len(labels):.1%})")
for clip, true, pred in zip(clips, labels, pred_labels):
    mark = "ok" if true == pred else "MISS"
    print(f"{mark}  {clip.name}  true={true}  pred={pred}")

print("Done")